# [LAB] A2A로 Researcher에게 일 맡기기

검색 도구를 쓰는 Researcher Agent를 A2A 서버로 띄우고, 이 노트북을 클라이언트로 삼아 조사를 맡깁니다. 받은 조사 결과로는 이 노트북의 모델이 블로그 글을 씁니다.

- **2장 Researcher Agent를 A2A 서버로 띄우기** (과제): 셀들이 서버 파일을 씁니다. 빈칸 다섯 곳을 AI와 채우고, 터미널에서 서버를 띄웁니다.
- **3장 Researcher에게 일을 맡기는 클라이언트** (과제): 요청 메시지를 만드는 빈칸 한 곳을 채우고, Agent Card 확인 → 위임 → 글쓰기를 차례로 실행합니다.

셀은 위에서부터 차례로 실행합니다. 서버는 터미널에서 켜 둔 채로 3장을 진행합니다. 요청 한 번에 모델을 두세 번 호출하므로 결과가 나오기까지 수십 초가 걸립니다.

In [ ]:
%cd ../../../course/Chapter02/notebooks

**목차**

- 1. 실습 방법
- 2. Researcher Agent를 A2A 서버로 띄우기 (예제) (과제)
  - 2.1 불러오기 (예제)
  - 2.2 검색 도구와 LLM 준비 (과제)
  - 2.3 agent 노드 (과제)
  - 2.4 그래프 연결과 컴파일 (과제)
  - 2.5 A2A AgentExecutor (과제)
  - 2.6 Agent Card (예제)
  - 2.7 Starlette 앱 조립 (예제)
  - 2.8 AI에게 물어보기
- 3. Researcher에게 일을 맡기는 클라이언트 (예제) (과제)
  - 3.1 call_researcher 함수 (과제)
  - 3.2 Agent Card 확인 (예제)
  - 3.3 Researcher에게 위임 (예제)
  - 3.4 Writer가 글 작성 (예제)
  - 3.5 AI에게 물어보기
- 4. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 강사에게 알립니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 강사에게 알립니다.

<details><summary>강사용 메모</summary>

- 원본: `course/Chapter02/code/raw/02-a2a/` 두 파일(raw). 설치된 a2a-sdk 1.1.5 기준입니다.
- 실행 검증(2026-09-28, `COURSE_ENV=out`, OpenAI): 빈칸을 정답으로 채운 사본으로 서버를 띄우고 새 커널로 노트북 전체를 실행해 오류 0이었습니다. 서버는 `.venv/bin/python course/Chapter02/notebooks/a2a_server.py`로 띄웠습니다.
- 수업에서는 서버 파일을 고칠 때마다 "첫 셀부터 다시 실행 → 터미널 Ctrl+C → 다시 띄우기"가 필요합니다. 이 과정이 실습 시간의 큰 몫입니다.
- 사내 확인 필요: 사내 게이트웨이의 `bind_tools` 도구 호출, 응답 시간(60초 timeout), 강의장 PC에서 `127.0.0.1` 접속, VS Code 터미널이 저장소 맨 위에서 열리는지.
- 생성되는 `course/Chapter02/notebooks/a2a_server.py`는 `.gitignore`의 `course/Chapter02/notebooks/*.py`로 Git에서 빠집니다.

</details>

## 1. 실습 방법

실습은 두 종류입니다. 제목 옆 표시와 코드 주석으로 구분합니다.

- **예제** `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- **과제** `# ✍️ WRITE_HERE:` 아래 빈칸(`...`)을 AI와 함께 채웁니다. **프롬프트 힌트**를 보고 지시를 직접 씁니다.

실행한 뒤에는 **성공 기준**과 비교해 스스로 판단합니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. 과제는 그다음 접힌 "정답 보기"와 비교합니다.

## 2. Researcher Agent를 A2A 서버로 띄우기 <small>(예제) (과제)</small>

검색 도구를 쓰는 작은 LangGraph Agent(Researcher)를 만들고, 다른 Agent가 요청할 수 있도록 A2A 서버로 감쌉니다. 서버는 노트북 셀에서 돌리지 않고 파일로 쓴 뒤 터미널에서 띄웁니다.

아래 셀은 모두 서버 파일 `a2a_server.py`의 한 부분입니다. 첫 셀이 파일을 새로 쓰고(`%%writefile`), 다음 셀부터는 뒤에 이어 씁니다(`%%writefile -a`). 그래서 **위에서부터 차례로** 실행해야 하고, 한 셀을 고쳤다면 **첫 셀부터 다시** 실행해 파일을 새로 씁니다.

셀을 실행하면 파일만 써지고 코드는 아직 실행되지 않습니다. 빈칸을 제대로 채웠는지는 서버를 띄우고 3장의 클라이언트로 요청을 보내야 보입니다.

**왜 배우나:** 안쪽 Agent는 지금까지 만든 LangGraph Agent와 같습니다. 여기에 AgentExecutor, Agent Card, RequestHandler 세 가지를 더하면 다른 Agent가 요청할 수 있는 서버가 됩니다.

<details><summary>강사용 메모</summary>

- 원본: `course/Chapter02/code/raw/02-a2a/01_a2a_server.py`(raw). `%%writefile`로 `course/Chapter02/notebooks/a2a_server.py`를 쓰고 터미널에서 `uv run python course/Chapter02/notebooks/a2a_server.py`로 띄웁니다.
- 노트북에서만 손본 곳(patch, 원본 파일은 그대로):
  - docstring의 실행·curl 안내를 뺐습니다(실행 안내는 터미널 안내 셀이 맡습니다).
  - `search_web` 안에 `print(f"[search_web] 검색어: {query}")` 한 줄을 더했습니다. 원본은 도구 호출이 보이지 않아, 조건부 엣지 빈칸을 잘못 채워도 답이 그럴듯하게 나와 조용히 틀립니다(목차 질문 1, 사용자 결정).
  - protobuf·gRPC 설명 주석 3줄을 한 줄로 줄였고, `# t.Message: protobuf 기반 응답 메시지`를 `A2A 응답 메시지`로 바꿨습니다(목차 질문 3).
  - 예제로 남긴 TODO 4개(Message 발행, Agent Card, RequestHandler, 라우트)는 같은 내용의 설명 주석으로 바꿨습니다.
  - Card `description`과 skill `examples` 둘째 항목은 솔루션 문구로 바꿨습니다(슬라이드 p7·p11 문구, 영문 표기).
- 원본 서버 터미널에는 HTTP 접속 로그(uvicorn)만 나옵니다. `[search_web]` 줄은 patch로 더한 것입니다.

</details>

### 2.1 불러오기 <small>(예제)</small>

In [ ]:
%%writefile a2a_server.py
"""
A2A 서버: LangGraph Agent를 A2A 프로토콜로 노출하기

A2A(Agent-to-Agent)는 Google이 제안한 에이전트 간 통신 표준이다.
에이전트를 HTTP 서비스로 노출하면 프레임워크에 관계없이 (LangGraph, CrewAI, AutoGen 등)
서로 통신할 수 있다.
"""

import asyncio
import uuid
from typing import Annotated

import a2a.types as t
import uvicorn
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers.default_request_handler_v2 import (
    DefaultRequestHandlerV2,
)
from a2a.server.routes.agent_card_routes import create_agent_card_routes
from a2a.server.routes.jsonrpc_routes import create_jsonrpc_routes
from a2a.server.tasks import InMemoryTaskStore
from a2a.server.events.in_memory_queue_manager import InMemoryQueueManager
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from starlette.applications import Starlette
from starlette.routing import Route
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

**성공 기준**

- 🟢 셀 아래에 `Writing a2a_server.py` 또는 `Overwriting a2a_server.py` 한 줄이 출력되었다

### 2.2 검색 도구와 LLM 준비 <small>(과제)</small>

**✍️ WRITE_HERE: search_web 도구를 LLM과 ToolNode에 연결해 주세요.**

Researcher가 쓸 모델과 도구 실행 노드를 준비합니다. 모델에는 `search_web` 도구를 연결해 도구 호출을 요청할 수 있게 하고, 그 요청을 실제로 실행할 `ToolNode`도 같은 도구로 만듭니다. 아래 셀에서 쓸 이름은 `llm`과 `tool_node`입니다.

**프롬프트 힌트** 이 셀을 붙여 넣고, `get_llm()`으로 받은 모델에 `search_web`을 연결한 `llm`과 같은 도구로 만든 `tool_node` 두 줄을 채워 달라고 합니다. `%%writefile` 줄과 빈칸 밖의 줄은 고치지 말라고 적습니다.

**성공 기준**

- 🟢 서버를 띄울 때 오류 없이 켜진다
- 🟢 3장에서 위임했을 때 서버 터미널에 `[search_web] 검색어: …` 줄이 나온다(모델이 검색 없이 답하면 안 나올 수 있으니 한두 번 더 위임해 봅니다)
- ⛔ 서버를 띄우자마자 터미널에 `NameError: name 'tool_node' is not defined`(또는 `llm`)가 나오고 끝난다 → 빈칸이 남았거나 이름이 다릅니다

In [ ]:
%%writefile -a a2a_server.py
# ── 내부 LangGraph Agent ──────────────────────────────────────────────────────


@tool
def search_web(query: str) -> str:
    """웹에서 정보를 검색한다."""
    print(f"[search_web] 검색어: {query}")
    return f'"{query}" 검색 결과: 해당 주제에 대한 최신 정보 (데모 데이터)'


class State(TypedDict):
    messages: Annotated[list, add_messages]


# ✍️ WRITE_HERE: search_web 도구를 LLM과 ToolNode에 연결해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
llm = get_llm().bind_tools([search_web])
tool_node = ToolNode([search_web])
```

</details>

### 2.3 agent 노드 <small>(과제)</small>

**✍️ WRITE_HERE: State의 messages를 도구가 연결된 LLM에 전달해 주세요.**

`agent_node`는 Researcher가 한 번 생각하는 단계입니다. 지금까지 쌓인 메시지를 도구가 연결된 `llm`에 보내고, 받은 응답을 상태의 `messages`에 더하도록 돌려줍니다.

**프롬프트 힌트** `agent_node` 함수를 붙여 넣고, 상태의 메시지 목록을 `llm`에 보내 받은 응답을 `messages` 키의 딕셔너리로 돌려 달라고 합니다. 함수 밖은 고치지 말라고 적습니다.

**성공 기준**

- 🟢 3장에서 위임했을 때 리서치 결과 글이 나온다
- ⛔ `리서치 결과:` 아래에 보낸 주제만 그대로 나온다 → 모델의 응답을 `messages`에 담아 돌려주는지 봅니다

In [ ]:
%%writefile -a a2a_server.py
def agent_node(state: State) -> dict:
    # ✍️ WRITE_HERE: State의 messages를 도구가 연결된 LLM에 전달해 주세요.
    ...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
return {"messages": [llm.invoke(state["messages"])]}
```

</details>

### 2.4 그래프 연결과 컴파일 <small>(과제)</small>

**✍️ WRITE_HERE: tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.**

노드 두 개(`agent`, `tools`)와 시작 엣지는 이미 있습니다. agent 다음에는 도구 호출 요청이 있으면 tools로, 없으면 끝으로 가게 나누고, tools를 실행한 뒤에는 다시 agent로 돌아오게 잇습니다. 이 연결이 빠져도 답은 그럴듯하게 나올 수 있습니다. 그래서 서버 터미널의 `[search_web]` 줄로 도구가 실제로 호출되었는지 확인합니다.

**프롬프트 힌트** 이 셀을 붙여 넣고, 위 설명의 두 가지 흐름을 엣지로 연결해 달라고 합니다. 갈림길에는 LangGraph가 제공하는 `tools_condition`을 쓰고 노드 이름은 바꾸지 말라고 적습니다.

**성공 기준**

- 🟢 3장에서 위임했을 때 서버 터미널에 `[search_web] 검색어: …` 줄이 나오고, 리서치 결과 글이 나온다
- ⛔ `리서치 결과:` 아래가 비어 있다 → 모델이 도구 호출을 요청했는데 tools로 가지 못하고 끝났습니다
- ⛔ 두세 번 위임해도 서버 터미널에 `[search_web]` 줄이 한 번도 나오지 않는다 → 도구를 거치지 않습니다. 조건부 엣지를 다시 봅니다

**✍️ WRITE_HERE: LangGraph Agent를 컴파일해 주세요.**

연결을 마친 그래프를 실행할 수 있는 Agent로 만듭니다. 아래 AgentExecutor가 이 Agent를 `langgraph_agent`라는 이름으로 호출합니다.

**프롬프트 힌트** 그래프를 컴파일해 `langgraph_agent`라는 이름에 담아 달라고 합니다. 한 줄이면 된다고 적습니다.

**성공 기준**

- 🟢 서버를 띄울 때 오류 없이 켜지고, 3장에서 위임했을 때 리서치 결과 글이 나온다
- ⛔ 3장에서 위임하면 오류가 나고 `name 'langgraph_agent' is not defined`가 보인다 → 빈칸이 남았거나 이름이 다릅니다

In [ ]:
%%writefile -a a2a_server.py
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)
builder.add_edge(START, "agent")
# ✍️ WRITE_HERE: tools_condition으로 도구 호출 여부를 분기하고 tools 이후 agent로 돌아오게 연결해 주세요.
...  # ✍️ 여기를 채웁니다
# ✍️ WRITE_HERE: LangGraph Agent를 컴파일해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")
```

```python
langgraph_agent = builder.compile()
```

</details>

### 2.5 A2A AgentExecutor <small>(과제)</small>

**✍️ WRITE_HERE: 내부 LangGraph Agent를 비동기로 호출하고 최종 답변을 추출해 주세요.**

AgentExecutor는 A2A 요청을 실제 Agent 실행으로 잇습니다. 요청 글(`user_text`)을 사람 메시지로 감싸 `langgraph_agent`를 **비동기로** 실행하고, 결과 메시지 목록의 마지막 내용을 `answer`에 담습니다. 바로 아래 코드가 `answer`를 A2A Message로 감싸 돌려보냅니다.

**프롬프트 힌트** `execute` 함수를 붙여 넣고, `user_text`를 `HumanMessage`로 감싸 `langgraph_agent`를 `await`로 비동기 호출한 뒤 마지막 메시지의 내용을 `answer`에 담아 달라고 합니다. 빈칸 밖은 고치지 말라고 적습니다.

**성공 기준**

- 🟢 3장에서 위임했을 때 리서치 결과 글이 나온다
- ⛔ 3장에서 위임하면 오류가 나고 `name 'answer' is not defined`가 보인다 → 빈칸이 남았거나 이름이 다릅니다
- ⛔ `await` 없이 `invoke`로 썼다 → 비동기 함수 안이므로 `ainvoke`와 `await`로 다시 씁니다

In [ ]:
%%writefile -a a2a_server.py
# ── A2A AgentExecutor ─────────────────────────────────────────────────────────
# AgentExecutor.execute(): RequestContext에서 사용자 메시지를 읽어
# EventQueue에 결과(Message)를 발행한다.
#
# a2a-sdk 1.x는 a2a.types의 타입(t.Message, t.Part 등)으로 메시지를 만든다.


class ResearcherExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # context.get_user_input(): 사용자 메시지의 텍스트 파트를 모아 문자열로 반환
        user_text = context.get_user_input() or ""

        # ✍️ WRITE_HERE: 내부 LangGraph Agent를 비동기로 호출하고 최종 답변을 추출해 주세요.
        ...  # ✍️ 여기를 채웁니다

        # t.Message: A2A 응답 메시지
        # t.Part(text=...): 텍스트 파트 (1.x에서는 TextPart 대신 Part.text 직접 사용)
        # 결과를 A2A Message로 감싸 EventQueue에 발행한다
        await event_queue.enqueue_event(
            t.Message(
                role=t.Role.ROLE_AGENT,
                parts=[t.Part(text=answer)],
                task_id=context.task_id,
                context_id=context.context_id,
                message_id=str(uuid.uuid4()),
            )
        )

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass

<details><summary>정답 보기</summary>

```python
result = await langgraph_agent.ainvoke(
    {"messages": [HumanMessage(content=user_text)]}
)
answer = result["messages"][-1].content
```

</details>

### 2.6 Agent Card <small>(예제)</small>

**✍️ CHANGE_HERE: Agent Card의 설명 문구를 바꿔 봅니다**

Agent Card는 다른 Agent가 가장 먼저 읽는 자기소개입니다. 처음에는 그대로 두고 끝까지 진행합니다. 3장까지 해 본 뒤 돌아와 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 따옴표 안 문구를 바꿉니다(예: "주제를 조사해 핵심만 세 줄로 정리합니다."). 바꾼 뒤에는 **첫 셀부터 이 장의 셀을 다시 실행**하고, 서버 터미널에서 Ctrl+C로 끈 뒤 다시 띄웁니다.

In [ ]:
%%writefile -a a2a_server.py
# ── Agent Card ────────────────────────────────────────────────────────────────
# Agent Card: 이 에이전트가 무엇을 할 수 있는지 설명하는 메타데이터.
# 클라이언트는 /.well-known/agent-card.json에서 이를 읽어 에이전트를 발견(discover)한다.

# 이름, 설명, 입출력 모드, capability, skill 정보를 적는다
agent_card = t.AgentCard(
    name="Researcher Agent",
    # ✍️ CHANGE_HERE: Agent Card의 설명 문구를 바꿔 봅니다
    description="주어진 주제를 웹에서 조사하고 요약 정보를 제공합니다.",
    version="1.0.0",
    capabilities=t.AgentCapabilities(streaming=False),
    default_input_modes=["text"],
    default_output_modes=["text"],
    supported_interfaces=[t.AgentInterface(url="http://localhost:8001")],
    skills=[
        t.AgentSkill(
            id="web_research",
            name="웹 리서치",
            description="주제를 검색해 핵심 정보를 수집하고 정리합니다.",
            tags=["research", "web"],
            examples=["AI Agent 기술 트렌드 조사", "LangGraph 최신 기능 알려줘"],
        )
    ],
)

### 2.7 Starlette 앱 조립 <small>(예제)</small>

In [ ]:
%%writefile -a a2a_server.py
# ── Starlette 앱 조립 ─────────────────────────────────────────────────────────

queue_manager = InMemoryQueueManager()
task_store = InMemoryTaskStore()

request_handler = DefaultRequestHandlerV2(
    # AgentExecutor, TaskStore, AgentCard, QueueManager를 연결한다
    agent_executor=ResearcherExecutor(),
    task_store=task_store,
    queue_manager=queue_manager,
    agent_card=agent_card,
)

# Agent Card 조회 라우트와 JSON-RPC 라우트를 만든다
routes: list[Route] = [
    *create_agent_card_routes(agent_card),
    *create_jsonrpc_routes(request_handler, rpc_url="/"),
]

app = Starlette(routes=routes)

if __name__ == "__main__":
    print("Researcher A2A 서버 시작: http://localhost:8001")
    print("Agent Card: http://localhost:8001/.well-known/agent-card.json")
    uvicorn.run(app, host="0.0.0.0", port=8001)

**성공 기준**

- 🟢 셀 아래에 `Appending to a2a_server.py` 한 줄이 출력되었다

위 셀을 실행하면 이 폴더에 서버 파일이 만들어집니다(빈칸을 먼저 채웁니다). 서버는 터미널에서 띄웁니다.

1. VS Code 메뉴 **터미널 → 새 터미널**을 엽니다(저장소 맨 위 폴더에서 열립니다).
2. 아래 명령을 붙여 넣고 실행합니다.

```
uv run python course/Chapter02/notebooks/a2a_server.py
```

3. 서버 시작 문구가 나오고 명령이 끝나지 않고 기다리면 켜진 것입니다. 이 터미널은 닫지 말고 노트북으로 돌아옵니다.
4. 서버 파일을 고쳤다면 터미널에서 **Ctrl+C**로 끄고, 위 셀을 다시 실행한 뒤 같은 명령으로 다시 띄웁니다.

**볼 지점**
- 서버를 띄운 터미널에 먼저 아래 두 줄이 나옵니다. 서버 파일 맨 끝의 `print`가 출력한 것입니다. 그 아래에 서버 프로그램(uvicorn)이 `INFO:`로 시작하는 줄을 이어서 출력하고, 마지막 줄이 `Uvicorn running on`으로 시작합니다.

```
Researcher A2A 서버 시작: http://localhost:8001
Agent Card: http://localhost:8001/.well-known/agent-card.json
```

- 시작 문구 위에 `queue_manager`가 쓰이지 않는다는 `DeprecationWarning`이 나옵니다. 서버 동작에는 영향이 없으니 넘어갑니다.
- 서버는 요청이 올 때까지 기다리기만 합니다. 모델은 요청이 들어올 때 호출됩니다.
- 3장에서 요청을 보내면 이 터미널에 `"POST / HTTP/1.1" 200 OK`로 끝나는 접속 기록이 나옵니다. 모델이 검색하기로 하면 그 앞에 `[search_web] 검색어: …` 줄도 나옵니다. 검색 여부는 모델이 정하므로 없는 때도 있습니다.
- 클라이언트에서 오류가 나면 **이 터미널부터** 봅니다. 서버 코드에서 난 오류(Traceback)가 여기에 나옵니다.

**성공 기준**

- 🟢 터미널에 `Uvicorn running on`으로 시작하고 `:8001`로 끝나는 주소가 적힌 줄이 나오고, 명령이 끝나지 않고 기다린다
- ⛔ `NameError`·`SyntaxError`가 나오고 바로 끝난다 → 빈칸을 확인하고, 첫 셀부터 다시 실행한 뒤 다시 띄웁니다
- ⛔ `address already in use`가 나온다 → 8001 포트를 쓰는 서버가 이미 켜져 있습니다. 앞서 띄운 서버 터미널에서 Ctrl+C로 끄고 다시 띄웁니다

<details><summary>강사용 메모</summary>

- 실측(2026-09-28, 사외 OpenAI): 서버 시작 때 `DeprecationWarning: A queue_manager was passed to DefaultRequestHandlerV2, but it is not used…`가 두 번 나온 뒤 시작 문구 2줄, uvicorn `INFO:` 4줄이 나옵니다. a2a-sdk 1.1.5에서 V2 핸들러가 queue_manager를 쓰지 않아 생기는 경고입니다. 원본 코드·슬라이드 p12가 queue_manager를 넘기므로 그대로 두었습니다.
- 빈칸을 남긴 채 띄워 본 결과(사외): ① 빈칸 → 시작하자마자 `NameError: name 'tool_node' is not defined`로 종료. ② 빈칸 → 오류 없이 보낸 주제가 그대로 돌아옴(`agent_node`가 None을 돌려 상태가 안 바뀜). ③ 빈칸 → 모델이 검색을 요청하면 빈 문자열이 돌아오고, 안 하면 정상처럼 보임. ④·⑤ 빈칸 → 서버 터미널에 Traceback, 클라이언트는 `InternalError`(메시지 `name '…' is not defined`).
- 정답 상태에서 위임 6회 중 4회 `search_web`을 호출했습니다(시스템 프롬프트 없음, 검색어는 영어로 만들기도 함). 그래서 `[search_web]` 줄은 "한두 번 더 위임해 봅니다"로 적었습니다.
- 캡스톤 `mcp_server.py`도 8001 포트를 씁니다. 두 서버를 동시에 띄우지 않습니다.

</details>

### 2.8 AI에게 물어보기

**프롬프트 힌트** `ResearcherExecutor`, `request_handler`, `routes` 셀을 붙여 넣고, 요청 한 건이 들어와 응답이 나갈 때까지 어떤 객체를 어떤 순서로 거치는지 설명해 달라고 합니다. 코드는 고치지 말라고 적습니다.

**성공 기준**

- 🟢 라우트(JSON-RPC) → RequestHandler → AgentExecutor → 내부 LangGraph Agent → EventQueue 순서를 짚었다
- 🟢 Agent Card 라우트는 카드 조회용이고 작업 요청은 JSON-RPC 라우트로 들어온다는 점을 짚었다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말고 설명만 해 달라고 다시 적습니다
- ⛔ `A2AStarletteApplication` 같은 이 코드에 없는 이름으로 설명한다 → 붙여 넣은 코드에 있는 이름으로만 설명해 달라고 합니다

## 3. Researcher에게 일을 맡기는 클라이언트 <small>(예제) (과제)</small>

이제 노트북이 클라이언트가 되어, 터미널에 띄운 Researcher 서버에 A2A로 조사를 맡깁니다. 순서는 슬라이드와 같습니다. Agent Card를 읽고, 그 정보로 Client를 만들고, 요청을 `t.Message`에 담아 보냅니다. 받은 조사 결과로는 이 노트북의 모델(Writer)이 블로그 글을 씁니다.

**이 장을 시작하기 전에 2장의 서버가 터미널에서 켜져 있어야 합니다.**

**왜 배우나:** 조사는 원격 Agent에게 맡기고 글쓰기는 여기서 직접 하는 분업입니다. 맡기는 쪽은 Researcher가 안에서 어떻게 만들어졌는지 몰라도 됩니다.

준비 코드입니다. A2A 클라이언트 도구를 불러오고, Researcher 서버 주소(`RESEARCHER_URL`)를 정합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
import asyncio
import uuid

import a2a.types as t
import httpx
from a2a.client.base_client import BaseClient
from a2a.client.card_resolver import A2ACardResolver
from a2a.client.client import ClientConfig
from a2a.client.transports.jsonrpc import JsonRpcTransport
from a2a.utils.constants import PROTOCOL_VERSION_1_0, VERSION_HEADER
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage

from config.llm import get_llm

load_dotenv()

RESEARCHER_URL = "http://127.0.0.1:8001"

<details><summary>강사용 메모</summary>

- 원본: `course/Chapter02/code/raw/02-a2a/02_a2a_client.py`(raw). 노트북에서만 손본 곳:
  - docstring(실행 순서 안내)을 뺐습니다.
  - `RESEARCHER_URL`을 솔루션 값 `http://127.0.0.1:8001`로 바꿨습니다. 서버는 `0.0.0.0`(IPv4)에 뜨는데 `localhost`가 IPv6(`::1`)로 먼저 풀리는 PC에서는 연결이 늦거나 실패할 수 있습니다(강의장 Windows PC 미확인).
  - 예제로 남긴 TODO 2개(CardResolver, transport·client)는 바로 위에 같은 뜻의 번호 주석(`# 1. …`, `# 2. …`)이 있어 지웠습니다.
  - `main()`을 셀 세 개(Agent Card 확인 / 위임 / Writer)로 풀어 top-level `await`로 실행합니다(목차 질문 2, 사용자 결정). `asyncio.run(main())`은 노트북에서 이미 이벤트 루프가 돌고 있어 쓰지 않습니다.
- 원본 코드에는 Task 조회·완료 처리 코드가 없습니다(커리큘럼 3.3 항목과 다름). 받은 내용을 유지해 더하지 않았습니다.

</details>

### 3.1 call_researcher 함수 <small>(과제)</small>

**✍️ WRITE_HERE: t.SendMessageRequest를 생성해주세요.**

`call_researcher`는 Agent Card 조회 → Client 생성 → 메시지 전송 → 응답에서 텍스트 꺼내기를 한 함수에 담았습니다. 빈칸은 세 번째 단계입니다. 받은 `query`를 사용자 역할의 `t.Message`에 텍스트 파트로 담고, 메시지마다 고유한 id를 붙여 `SendMessageRequest`로 감싼 `req`를 만듭니다. 이 셀은 함수만 만들고 아무것도 출력하지 않습니다. 실제 호출은 아래 "Researcher에게 위임" 셀에서 합니다.

**프롬프트 힌트** 이 함수를 붙여 넣고, `query`를 사용자 역할의 `t.Message`에 담아 `t.SendMessageRequest`로 감싼 `req`를 만들어 달라고 합니다. a2a-sdk 1.x 방식으로, 같은 셀과 서버 코드에 이미 쓰인 타입(`t.Part`, `t.Role`)만 쓰라고 적습니다.

**성공 기준**

- 🟢 이 셀이 오류 없이 끝나고 아무것도 출력하지 않는다
- 🟢 아래 "Researcher에게 위임" 셀에서 리서치 결과 글이 나온다
- ⛔ "Researcher에게 위임" 셀에서 `NameError: name 'req' is not defined`가 나온다 → 빈칸이 남았거나 이름이 다릅니다
- ⛔ `TextPart`·`MessageSendParams` 같은 이름에서 오류가 난다 → 예전 버전 방식입니다. 서버 코드의 `t.Message`와 같은 모양으로 다시 요청합니다

In [ ]:
async def call_researcher(query: str) -> str:
    """A2A 프로토콜로 Researcher Agent에게 조사를 요청한다.

    A2A-Version 헤더를 명시해야 서버가 1.x 형식 요청임을 인식한다.
    헤더가 없으면 서버는 레거시(0.3) 버전으로 가정하여 거부한다.
    """
    headers = {VERSION_HEADER: PROTOCOL_VERSION_1_0}
    async with httpx.AsyncClient(headers=headers, timeout=60.0) as http:
        # 1. Agent Card 조회: 에이전트가 어떤 기능을 제공하는지 확인
        resolver = A2ACardResolver(http, RESEARCHER_URL)
        card = await resolver.get_agent_card()

        # 2. JSON-RPC 트랜스포트로 클라이언트 생성
        transport = JsonRpcTransport(http, card, f"{RESEARCHER_URL}/")
        client = BaseClient(card, ClientConfig(), transport, [])

        # 3. 메시지 전송
        # ✍️ WRITE_HERE: t.SendMessageRequest를 생성해주세요.
        ...  # ✍️ 여기를 채웁니다

        async for response in client.send_message(req):
            # StreamResponse.message: 에이전트가 반환한 Message 객체
            if response.HasField("message"):
                return "".join(
                    part.text for part in response.message.parts if part.text
                )

    return "응답을 받지 못했습니다."

<details><summary>정답 보기</summary>

```python
req = t.SendMessageRequest(
    message=t.Message(
        role=t.Role.ROLE_USER,
        parts=[t.Part(text=query)],
        message_id=str(uuid.uuid4()),
    )
)
```

</details>

### 3.2 Agent Card 확인 <small>(예제)</small>

In [ ]:
# ── Agent Card 확인 ────────────────────────────────────────────────────────
async with httpx.AsyncClient() as http:
    resp = await http.get(f"{RESEARCHER_URL}/.well-known/agent-card.json")
    card = resp.json()

print("=== Researcher Agent Card ===")
print(f"이름   : {card['name']}")
print(f"설명   : {card['description']}")
print(f"스킬   : {[s['name'] for s in card.get('skills', [])]}")

**성공 기준**

- 🟢 `이름   : Researcher Agent`, `설명   : …`, `스킬   : ['웹 리서치']` 세 줄이 출력되었다
- 🟢 (2장에서 설명 문구를 바꿨다면) 설명 줄이 바꾼 문구로 나온다
- ⛔ `ConnectError`(연결 거부)가 난다 → 서버 터미널이 켜져 있는지 봅니다
- ⛔ 설명을 바꿨는데 예전 문구가 나온다 → 2장 셀을 첫 셀부터 다시 실행하고, 서버를 Ctrl+C로 끈 뒤 다시 띄웁니다

### 3.3 Researcher에게 위임 <small>(예제)</small>

**✍️ CHANGE_HERE: Researcher에게 맡길 주제를 바꿔 봅니다**

먼저 그대로 실행해 결과를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 따옴표 안 주제를 바꿔(예: "반도체 공정의 AI 활용 사례") 이 셀을 다시 실행합니다. 서버는 다시 띄우지 않아도 됩니다.

In [ ]:
# ── Researcher에게 조사 위임 ───────────────────────────────────────────────
# ✍️ CHANGE_HERE: Researcher에게 맡길 주제를 바꿔 봅니다
topic = "LangGraph Multi-Agent 시스템의 실제 활용 사례"
print(f'=== Researcher에게 위임: "{topic}" ===')
research_result = await call_researcher(topic)
print(f"리서치 결과:\n{research_result}\n")

**성공 기준**

- 🟢 `=== Researcher에게 위임: "…" ===` 아래에 `리서치 결과:`와 조사한 글이 출력되었다
- 🟢 서버 터미널에 `[search_web] 검색어: …` 줄이 새로 나왔다(모델이 검색 없이 답하면 없을 수 있으니 한 번 더 실행해 봅니다)
- 🟢 주제를 바꾸면 바꾼 주제에 맞는 글이 나온다
- ⛔ 오류가 나거나 `리서치 결과:` 아래가 비어 있다 → 서버 터미널에 나온 오류(Traceback)를 먼저 보고, 2장의 빈칸 성공 기준과 비교합니다

### 3.4 Writer가 글 작성 <small>(예제)</small>

In [ ]:
# ── Writer가 리서치 결과로 글 작성 ────────────────────────────────────────
# Writer는 직접 LLM을 쓰지만, 리서치 작업은 Researcher에게 위임했다.
# → 이종 Agent 간 역할 분리가 A2A의 핵심
print("=== Writer가 블로그 포스트 작성 ===")
llm = get_llm()
response = llm.invoke(
    [
        SystemMessage(
            content="너는 전문 작가야. 리서치 결과를 바탕으로 블로그 포스트를 작성해줘."
        ),
        HumanMessage(
            content=f"리서치 결과:\n{research_result}\n\n이 내용으로 블로그 포스트를 작성해줘."
        ),
    ]
)
print(response.content)

**성공 기준**

- 🟢 `=== Writer가 블로그 포스트 작성 ===` 아래에 리서치 결과를 바탕으로 한 블로그 글이 출력되었다
- ⛔ `NameError: name 'research_result' is not defined`가 난다 → 바로 위 셀을 먼저 실행합니다

**볼 지점**
- "Agent Card 확인" 셀은 모델을 호출하지 않습니다. 서버가 공개한 카드(`/.well-known/agent-card.json`)를 읽어 이름·설명·스킬을 보여 줄 뿐입니다.
- "Researcher에게 위임" 셀의 조사는 이 노트북이 아니라 **서버의 Agent**가 합니다. 그래서 검색 기록(`[search_web]` 줄)도 이 노트북이 아니라 서버 터미널에 나옵니다.
- "Writer가 글 작성" 셀은 서버와 관계없이 이 노트북에서 모델을 직접 호출합니다. 결과는 실행할 때마다 조금씩 다릅니다.

<details><summary>강사용 메모</summary>

- 실측(2026-09-28, 사외 OpenAI): 3장 셀 전체(카드 확인·위임·Writer)가 약 30초. 사내 게이트웨이에서 서버 쪽 LLM 2회 + 도구 호출이 httpx timeout 60초 안에 끝나는지는 미확인입니다. 넘으면 `timeout=60.0`을 patch로 늘립니다.
- 주제 CHANGE_HERE는 클라이언트 쪽 값이라 서버를 다시 띄우지 않아도 됩니다. 설명 CHANGE_HERE(2장)는 서버 재시작이 필요합니다.
- 클라이언트 오류는 a2a 클라이언트 예외(`InternalError` 등)로 올라옵니다. 원인은 서버 터미널의 Traceback에 있습니다.

</details>

### 3.5 AI에게 물어보기

**프롬프트 힌트** "Researcher에게 위임"과 "Writer가 글 작성" 두 셀을 붙여 넣고, 어느 줄이 원격 Agent에게 일을 맡기고 어느 줄이 이 노트북에서 직접 모델을 호출하는지 설명해 달라고 합니다. 코드는 고치지 말라고 적습니다.

**성공 기준**

- 🟢 `await call_researcher(topic)`은 A2A로 서버의 Researcher에게 맡기는 줄이라고 짚었다
- 🟢 `llm.invoke(...)`는 이 노트북에서 모델을 직접 호출하는 줄이라고 짚었다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말고 설명만 해 달라고 다시 적습니다

## 4. 정리

A2A로 Agent 사이에 일을 나눴습니다.

| 쪽 | 한 일 | 쓴 것 |
|---|---|---|
| 서버 (Researcher) | 요청을 받아 내부 LangGraph Agent를 실행하고 결과를 돌려줌 | AgentExecutor, Agent Card, RequestHandler |
| 클라이언트 (이 노트북) | 카드를 읽고, 조사를 맡기고, 받은 결과로 글을 씀 | A2ACardResolver, SendMessageRequest |

맡기는 쪽은 Researcher의 내부 구현을 몰라도 카드와 메시지 형식만 맞추면 일을 맡길 수 있습니다.

**실습을 마치면 서버 터미널에서 Ctrl+C로 서버를 끕니다.** 같은 8001 포트를 쓰는 다른 서버를 띄울 때 부딪히지 않게 하기 위해서입니다.